# Data Cleaning

In [65]:
import re
import warnings
import logging


import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

In [66]:
RAW_DATA_PATH= "./data.csv"
CLEANING_LOG_PATH="cleaning_log.csv"
CLEANING_LOGGING_PATH="cleaning.log"

# Prepare for logging

In [67]:
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler(CLEANING_LOGGING_PATH),   # writes to file
        logging.StreamHandler(),               # prints to console
    ],
)

cleaning_log: list[dict] = []

In [68]:
def log_cleaning_action(step: str,rule: str,records_affected: int,action: str,rationale: str) -> None:
    """
    Append one cleaning decision to the in-memory cleaning log.

    Args:
        step:The cleaning dimension
        rule:specific rule applied
        records_affected: Number of rows or values changed.
        action: action that was done.
        rationale: Why this action was chosen.
    """
    cleaning_log.append({
        "step": step,
        "rule": rule,
        "records_affected": records_affected,
        "action": action,
        "rationale": rationale,
    })
    logging.info(f"[LOG] {step} | {rule} | {records_affected} records | {action}")

### Load Raw Data

In [69]:
raw_df = pd.read_csv(RAW_DATA_PATH,low_memory=False)

# Work on a copy, raw_df is never modified
df=raw_df.copy()

print("DATA SHAPE")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print()

print("COLUMN NAMES")
print(df.columns.tolist())
print()

print("DTTYPES")
print(df.dtypes.value_counts())

DATA SHAPE
Rows: 37,003 | Columns: 70

COLUMN NAMES
['listing_id', 'internal_id', 'category', 'listing_type', 'detail_url', 'property_type', 'offering_type', 'completion_status', 'title', 'price_egp', 'price_period', 'price_currency', 'location_full', 'city', 'town', 'district', 'subdistrict', 'lat', 'lon', 'bedrooms', 'bathroom', 'area_value', 'area_unit', 'furnished', 'listing_level', 'is_premium', 'is_verified', 'is_featured', 'is_new_construction', 'is_direct_from_developer', 'is_exclusive', 'listed_date', 'images_count', 'has_video', 'video_url', 'reference', 'rera', 'description', 'amenities', 'payment_plan', 'agent_id', 'agent_name', 'agent_email', 'agent_is_verified', 'agent_languages', 'broker_id', 'broker_name', 'broker_email', 'broker_phone', 'contact_phone', 'contact_whatsapp', 'contact_email', 'scraped_at', 'source', 'id', 'url', 'title.1', 'scraped_at.1', 'dist_nearest_school_km', 'school_count_within_3km', 'dist_nearest_hospital_km', 'hospital_count_within_3km', 'dist_ne

# we will remove irrelevant columns to our problem before starting

In [70]:
irrelevant_columns_to_remove =[
    "listing_id","internal_id","detail_url","title","images_count",	"has_video","video_url","reference","description",
    "agent_id","agent_name","agent_email","agent_is_verified","agent_languages","broker_id","broker_name","broker_email",
    "broker_phone","contact_phone","contact_whatsapp","location_full","contact_email","scraped_at","source","id","url","title.1","scraped_at.1",
    "category", "listing_type"
]

print("Original number of Columns:" f"{df.shape[1]}")

df_relevance = df.drop(columns=irrelevant_columns_to_remove,errors="ignore")
print("Number of Columns after cleaning:" f"{df_relevance.shape[1]}")

Original number of Columns:70
Number of Columns after cleaning:40


In [71]:
df=df_relevance.copy()
df.columns

Index(['property_type', 'offering_type', 'completion_status', 'price_egp',
       'price_period', 'price_currency', 'city', 'town', 'district',
       'subdistrict', 'lat', 'lon', 'bedrooms', 'bathroom', 'area_value',
       'area_unit', 'furnished', 'listing_level', 'is_premium', 'is_verified',
       'is_featured', 'is_new_construction', 'is_direct_from_developer',
       'is_exclusive', 'listed_date', 'rera', 'amenities', 'payment_plan',
       'dist_nearest_school_km', 'school_count_within_3km',
       'dist_nearest_hospital_km', 'hospital_count_within_3km',
       'dist_nearest_supermarket_km', 'supermarket_count_within_3km',
       'dist_nearest_mall_km', 'mall_count_within_3km',
       'dist_nearest_transit_station_km', 'transit_station_count_within_3km',
       'dist_nearest_cafe_restaurant_km', 'cafe_restaurant_count_within_3km'],
      dtype='object')

## Step 1: Accuracy

### Rule-based Accuracy Corrections

### Quarantine Invalid Records

In [72]:
# create masks
area_mask = df["area_value"] > 1000
price_mask = df["price_egp"] > 20_000_000
lon_mask = (df["lon"] < 25.0) | (df["lon"] > 35.0)
lat_mask = (df["lat"] < 22.0) | (df["lat"] > 31.0)


rejection_mask = (
    area_mask |     # unrealistic area
    price_mask |  # unrealistic price
    lon_mask |  # invalid longitude
    lat_mask    # invalid latitude
)

df_cleaned =df[~rejection_mask].copy()
df_quarantined =df[rejection_mask].copy()



log_cleaning_action(step="accuracy",rule="area_value > 1000",records_affected=area_mask.sum(),
    action=f"quarantined non realistic {area_mask.sum()} records",
    rationale="non realistic area values likely indicate data entry errors"
)

log_cleaning_action(step="accuracy",rule="price_egp > 20_000_000",records_affected=price_mask.sum(),
    action=f"quarantined non realistic {price_mask.sum()} records",
    rationale="non realistic price values likely indicate data entry errors"
)

log_cleaning_action(step="accuracy",rule="lon out of range",records_affected=lon_mask.sum(),
    action=f"quarantined invalid {lon_mask.sum()} records",
    rationale="longitude values outside of Egypt's geographic range"
)

log_cleaning_action(step="accuracy",rule="lat out of range",records_affected=lat_mask.sum(),
    action=f"quarantined invalid {lat_mask.sum()} records",
    rationale="latitude values outside of Egypt's geographic range"
)

df_quarantined["rejection_reason"] = (
    df[rejection_mask].apply(
        lambda row: '; '.join([
            'area_value > 1000' if row['area_value'] > 1000 else '',
            'price_egp > 20M' if row['price_egp'] > 20_000_000 else '',
            'invalid_lon' if (row['lon'] < 25.0 or row['lon'] > 35.0) else '',
            'invalid_lat' if (row['lat'] < 22.0 or row['lat'] > 31.0) else '',
        ]).strip('; '),
        axis=1
    )
)

log_cleaning_action(step="accuracy",rule="total_quarantined",records_affected=rejection_mask.sum(),
    action=f"quarantined {rejection_mask.sum()} records with reasons logged",
    rationale="Allows for later review and potential recovery of records if needed"
)

# --- Final ---
df = df_cleaned

INFO:root:[LOG] accuracy | area_value > 1000 | 3 records | quarantined non realistic 3 records
INFO:root:[LOG] accuracy | price_egp > 20_000_000 | 876 records | quarantined non realistic 876 records
INFO:root:[LOG] accuracy | lon out of range | 0 records | quarantined invalid 0 records
INFO:root:[LOG] accuracy | lat out of range | 1632 records | quarantined invalid 1632 records
INFO:root:[LOG] accuracy | total_quarantined | 2480 records | quarantined 2480 records with reasons logged


### Accuracy Step Summary

In [73]:
accuracy_logs = [row for row in cleaning_log if row["step"] == "accuracy"]
print("ACCURACY SUMMARY\n")
print(f"Actions taken: {len(accuracy_logs)}")
print(f"Total records affected: {sum(row['records_affected'] for row in accuracy_logs):,}")
print(f"Records remaining: {len(df):,}")
print()
for i, row in enumerate(accuracy_logs, 1):
    print(f"{i}. {row['rule']}")
    action_name = row.get("action", row.get("action_taken", ""))
    print(f"   Records affected: {row['records_affected']:,} | Action: {action_name}")
    print()

ACCURACY SUMMARY

Actions taken: 5
Total records affected: 4,991
Records remaining: 34,523

1. area_value > 1000
   Records affected: 3 | Action: quarantined non realistic 3 records

2. price_egp > 20_000_000
   Records affected: 876 | Action: quarantined non realistic 876 records

3. lon out of range
   Records affected: 0 | Action: quarantined invalid 0 records

4. lat out of range
   Records affected: 1,632 | Action: quarantined invalid 1632 records

5. total_quarantined
   Records affected: 2,480 | Action: quarantined 2480 records with reasons logged



# consistency step

In [54]:
def show_value_counts(df,columns=None):
    for col in df.columns:
        print(f"\nColumn: {col}")
        print(df[col].value_counts(dropna=False))

In [55]:
df.columns

Index(['category', 'listing_type', 'property_type', 'offering_type',
       'completion_status', 'price_egp', 'price_period', 'price_currency',
       'city', 'town', 'district', 'subdistrict', 'lat', 'lon', 'bedrooms',
       'bathroom', 'area_value', 'area_unit', 'furnished', 'listing_level',
       'is_premium', 'is_verified', 'is_featured', 'is_new_construction',
       'is_direct_from_developer', 'is_exclusive', 'listed_date', 'rera',
       'amenities', 'payment_plan', 'dist_nearest_school_km',
       'school_count_within_3km', 'dist_nearest_hospital_km',
       'hospital_count_within_3km', 'dist_nearest_supermarket_km',
       'supermarket_count_within_3km', 'dist_nearest_mall_km',
       'mall_count_within_3km', 'dist_nearest_transit_station_km',
       'transit_station_count_within_3km', 'dist_nearest_cafe_restaurant_km',
       'cafe_restaurant_count_within_3km'],
      dtype='object')

In [56]:
df["rera"].value_counts()

Series([], Name: count, dtype: int64)

In [58]:
df["category"].value_counts()

category
Residential    24620
buy             9903
Name: count, dtype: int64

In [57]:
show_value_counts(df,df.columns)


Column: category
category
Residential    24620
buy             9903
Name: count, dtype: int64

Column: listing_type
listing_type
Apartments    24620
property       9903
Name: count, dtype: int64

Column: property_type
property_type
Apartments    24620
Apartment      9903
Name: count, dtype: int64

Column: offering_type
offering_type
for-sale                24620
Residential for Sale     9903
Name: count, dtype: int64

Column: completion_status
completion_status
completed             20188
under-construction     8876
off_plan_primary       2568
completed_primary      2380
off_plan                507
NaN                       4
Name: count, dtype: int64

Column: price_egp
price_egp
6000000.0     815
7000000.0     686
6500000.0     650
5000000.0     650
8000000.0     557
             ... 
14836120.0      1
4466400.0       1
8270000.0       1
5814367.0       1
8866000.0       1
Name: count, Length: 5205, dtype: int64

Column: price_period
price_period
NaN     24620
sell     9903
Name: cou

## Step 2: Consistency

### Standardize Capitalization

In [62]:
pd.set_option('display.max_rows', None)

In [63]:
df["town"].value_counts()


town
New Cairo                       7443
New Cairo City                  3758
Sheikh Zayed                    3409
6th of October                  2806
New Capital City                2147
Madinaty                        1848
Sheikh Zayed City               1733
Hurghada                        1473
Mostakbal City                  1238
6 October City                  1140
Shorouk City                     997
Hadayek October                  768
Sheraton                         558
Mostakbal City - Future City     511
New Heliopolis                   474
Nasr City                        423
North Coast                      388
Katameya                         371
Mokattam                         278
Hadayek al-Ahram                 240
Zahraa Al Maadi                  214
Badr City                        211
Obour City                       209
Soma Bay                         152
Mohandessin                      149
Hay El Maadi                     141
Maadi                            

In [42]:
TITLE_CASE_COLUMNS = ["district", "city", "town", "subdistrict", "location_full"]


def standardize_title_case_columns(
    df: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    """Standardize string columns to Title Case after stripping whitespace.

    Special handling: remove leading articles (e.g., "The") and canonicalize
    ordinal tokens so variants like "5Th" / "The 5Th Settlement" -> "5th Settlement".

    Args:
        df: Input DataFrame.
        columns: List of string column names to standardize.

    Returns:
        DataFrame with specified columns standardized to Title Case.
    """
    df = df.copy()

    def canonicalize_location(val: object) -> object:
        if pd.isna(val):
            return val
        s = str(val).strip()
        # Remove leading "The " (case-insensitive) and collapse spaces
        s = re.sub(r"^the\s+", "", s, flags=re.I)
        s = re.sub(r"\s+", " ", s)
        lower = s.lower()
        parts = re.split(r"\s+", lower)

        def fmt(tok: str) -> str:
            m = re.match(r"^(\d+)(st|nd|rd|th)?$", tok)
            if m:
                # keep numeric ordinals as digits + lowercase suffix (e.g., '5th')
                return (m.group(1) or "") + (m.group(2) or "")
            return tok.capitalize()

        return " ".join(fmt(p) for p in parts)

    for col in columns:
        if col not in df.columns:
            continue
        before = df[col].nunique()
        # Apply canonicalization while preserving NaNs
        df[col] = df[col].apply(canonicalize_location)
        after = df[col].nunique()
        log_cleaning_action(
            step="Consistency",
            rule=f"{col} -> canonical title/ordinal form",
            records_affected=int(df[col].notna().sum()),
            action="strip + remove leading 'The' + canonicalize ordinals",
            rationale=f"Unique values: {before} -> {after} after normalization",
        )
    return df


# Apply to title-case columns
df = standardize_title_case_columns(df, TITLE_CASE_COLUMNS)

# Spot-check
print("Sample district values after standardization:")
print(df["district"].value_counts().head(10))

[LOG] Consistency | district -> canonical title/ordinal form | 30170 records | strip + remove leading 'The' + canonicalize ordinals
[LOG] Consistency | city -> canonical title/ordinal form | 35547 records | strip + remove leading 'The' + canonicalize ordinals
[LOG] Consistency | town -> canonical title/ordinal form | 35547 records | strip + remove leading 'The' + canonicalize ordinals
[LOG] Consistency | subdistrict -> canonical title/ordinal form | 15427 records | strip + remove leading 'The' + canonicalize ordinals
[LOG] Consistency | location_full -> canonical title/ordinal form | 35547 records | strip + remove leading 'The' + canonicalize ordinals
Sample district values after standardization:
district
5th Settlement              8334
1st Settlement               957
Sheikh Zayed Compounds       900
Rehab City                   557
6 October Compounds          513
Mostakbal City Compounds     466
Sarai                        433
6th Settlement               380
O West Compound      

### Standardize Format for Categorical Columns (furnished, completion_status)

In [43]:
# mappings
FURNISHED_MAP = {
    "yes": "Furnished", "y": "Furnished", "1": "Furnished",
    "true": "Furnished", "furnished": "Furnished", "YES": "Furnished",
    "no": "Unfurnished", "n": "Unfurnished", "0": "Unfurnished",
    "false": "Unfurnished", "unfurnished": "Unfurnished", "NO": "Unfurnished",
    "semi": "Semi-Furnished", "semi-furnished": "Semi-Furnished",
    "semi furnished": "Semi-Furnished", "PARTLY": "Semi-Furnished",
}

COMPLETION_STATUS_MAP = {
    "ready": "Ready", "completed": "Ready", "finished": "Ready",
    "under construction": "Under Construction", "off plan": "Off Plan",
    "offplan": "Off Plan", "off_plan": "Off Plan", "resale": "Resale",
}


def standardize_categorical_column(
    df: pd.DataFrame,
    column: str,
    mapping: dict[str, str],
) -> pd.DataFrame:
    """Map all variants of a categorical column to canonical labels.

    Values not in the mapping are left as-is (they may already be canonical
    or will be handled by missing value imputation in a later step).

    Args:
        df: Input DataFrame.
        column: Column name to standardize.
        mapping: Dict mapping raw lowercase variants to canonical labels.

    Returns:
        DataFrame with the specified column standardized.
    """
    df = df.copy()
    # Normalize input to lowercase for matching, then apply mapping
    normalized = df[column].str.strip().str.lower()
    mapped = normalized.map(mapping)
    # Only replace where the mapping has a match
    changed_mask = mapped.notna() & (df[column] != mapped)
    df.loc[changed_mask, column] = mapped[changed_mask]
    log_cleaning_action(
        step="Consistency",
        rule=f"{column} -> canonical labels",
        records_affected=int(changed_mask.sum()),
        action="Map to canonical form",
        rationale=f"Unify {len(mapping)} variants into consistent labels",
    )
    return df


# Standardize furnished
df = standardize_categorical_column(df, "furnished", FURNISHED_MAP)

# Standardize completion_status
df = standardize_categorical_column(df, "completion_status", COMPLETION_STATUS_MAP)

# Standardize listing_level: should be lowercase "standard" / "premium"
df["listing_level"] = df["listing_level"].str.strip().str.lower()

print("furnished unique values:", df["furnished"].unique())
print("completion_status unique values:", df["completion_status"].unique())
print("listing_level unique values:", df["listing_level"].unique())

[LOG] Consistency | furnished -> canonical labels | 27708 records | Map to canonical form
[LOG] Consistency | completion_status -> canonical labels | 22298 records | Map to canonical form
furnished unique values: <StringArray>
['PARTLY', nan, 'Furnished', 'Unfurnished']
Length: 4, dtype: str
completion_status unique values: <StringArray>
[          'Off Plan',              'Ready',  'completed_primary',
   'off_plan_primary',                  nan, 'under-construction']
Length: 6, dtype: str
listing_level unique values: <StringArray>
['premium', 'featured', 'standard', 'hot', 'superhot']
Length: 5, dtype: str


### Standardize Format of Amenities Column

In [44]:
def parse_and_standardize_amenities(amenity_str: object) -> str:
    """Parse a raw amenities string into a sorted, lowercase, comma-separated form.

    Examples:
        "[Pool, Balcony, Security]" -> "balcony, pool, security"
        "Pool|Gym"                  -> "gym, pool"
        NaN                         -> ""

    Args:
        amenity_str: Raw amenities value (string or NaN).

    Returns:
        Canonical amenities string, or empty string if null/unparseable.
    """
    if pd.isna(amenity_str):
        return ""

    raw = str(amenity_str)
    # Remove brackets
    raw = re.sub(r"[\[\]{}]", "", raw)
    # Split on comma or pipe
    items = re.split(r"[,|;]", raw)
    # Clean each item: strip, lowercase, remove empty strings
    items = [item.strip().lower() for item in items if item.strip()]
    # Sort for consistency and rejoin
    return ", ".join(sorted(set(items)))


before = df["amenities"].isna().sum()
df["amenities"] = df["amenities"].apply(parse_and_standardize_amenities)
after = (df["amenities"] == "").sum()

log_cleaning_action(
    step="Consistency",
    rule="amenities -> sorted lowercase comma-separated string",
    records_affected=int(df["amenities"].notna().sum()),
    action="Parse brackets, split, lowercase, sort, rejoin",
    rationale="Canonical form required for reproducible amenity_count and has_luxury features",
)
print(f"Originally NaN: {before:,} | Now empty string: {after:,}")
print("Sample:", df["amenities"].dropna().head(5).tolist())

[LOG] Consistency | amenities -> sorted lowercase comma-separated string | 35547 records | Parse brackets, split, lowercase, sort, rejoin
Originally NaN: 26,016 | Now empty string: 26,016
Sample: ["balcony, built in wardrobes, central a/c, children's pool, covered parking, lobby in building, security, shared gym, shared pool, shared spa, view of landmark", "balcony, built in wardrobes, central a/c, children's pool, covered parking, kitchen appliances, lobby in building, maids room, private garden, private pool, security, shared gym, shared pool, shared spa, study, view of landmark, view of water, walk-in closet", "balcony, built in wardrobes, central a/c, children's pool, covered parking, kitchen appliances, lobby in building, maids room, private garden, private pool, security, shared gym, shared pool, shared spa, study, view of landmark, view of water, walk-in closet", 'balcony, built in wardrobes, central a/c, security', 'balcony, built in wardrobes, covered parking, lobby in buildin

### Coerce Types for Numerical and Bool Columns

In [45]:
def coerce_bedrooms_bathrooms(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce bedrooms and bathrooms to nullable integer, handling string variants.

    String mappings:
    - "studio" -> 0
    - "7+" -> 7  (take the numeric floor)
    - Other strings -> extract leading digits, else NaN

    Args:
        df: Input DataFrame.

    Returns:
        DataFrame with bedrooms and bathrooms as Int64 (nullable integer).
    """
    df = df.copy()
    if "bathroom" in df.columns and "bathrooms" not in df.columns:
        df = df.rename(columns={"bathroom": "bathrooms"})
    room_cols = [col for col in ["bedrooms", "bathrooms"] if col in df.columns]
    for col in room_cols:

        def parse_room_value(val: object) -> object:
            if pd.isna(val):
                return pd.NA
            s = str(val).strip().lower()
            if s == "studio":
                return 0
            # Extract leading number (handles "7+", "3+", "2 rooms", etc.)
            match = re.match(r"(\d+)", s)
            if match:
                return int(match.group(1))
            return pd.NA

        before_nulls = df[col].isna().sum()
        df[col] = df[col].apply(parse_room_value).astype("Int64")
        after_nulls = df[col].isna().sum()
        log_cleaning_action(
            step="Consistency",
            rule=f"{col}: parse string variants (studio, 7+, etc.) -> Int64",
            records_affected=int(len(df) - before_nulls),
            action="Parse and cast to Int64",
            rationale=f"Mixed-type column; {after_nulls - before_nulls} new NAs from unparseable values",
        )
    return df


def coerce_boolean_columns(df: pd.DataFrame, bool_cols: list[str]) -> pd.DataFrame:
    """Coerce boolean-intended columns to proper bool dtype.

    Args:
        df: Input DataFrame.
        bool_cols: List of column names that should be boolean.

    Returns:
        DataFrame with specified columns cast to boolean.
    """
    df = df.copy()
    for col in bool_cols:
        if col not in df.columns:
            continue
        df[col] = df[col].astype("boolean")
        log_cleaning_action(
            "Consistency",
            f"{col} -> boolean",
            len(df),
            "astype('boolean')",
            "Enforce boolean dtype for flag column",
        )
    return df


df = coerce_bedrooms_bathrooms(df)
df = coerce_boolean_columns(df, BOOLEAN_COLUMNS)

# Coerce area and price to float (just to enforce dtype)
df["area_value"] = pd.to_numeric(df["area_value"], errors="coerce").astype("float64")
df["price_egp"] = pd.to_numeric(df["price_egp"], errors="coerce").astype("float64")

print("Dtypes after coercion:")
print(df.dtypes)

[LOG] Consistency | bedrooms: parse string variants (studio, 7+, etc.) -> Int64 | 35547 records | Parse and cast to Int64
[LOG] Consistency | bathrooms: parse string variants (studio, 7+, etc.) -> Int64 | 35546 records | Parse and cast to Int64
[LOG] Consistency | is_premium -> boolean | 35547 records | astype('boolean')
[LOG] Consistency | is_verified -> boolean | 35547 records | astype('boolean')
[LOG] Consistency | is_featured -> boolean | 35547 records | astype('boolean')
[LOG] Consistency | is_new_construction -> boolean | 35547 records | astype('boolean')
[LOG] Consistency | is_direct_from_developer -> boolean | 35547 records | astype('boolean')
[LOG] Consistency | is_exclusive -> boolean | 35547 records | astype('boolean')
[LOG] Consistency | has_video -> boolean | 35547 records | astype('boolean')
[LOG] Consistency | agent_is_verified -> boolean | 35547 records | astype('boolean')
Dtypes after coercion:
listing_id                              str
internal_id                    

### Consistency Step Summary

In [46]:
print("CONSISTENCY SUMMARY\n")
print(f"bedrooms dtype:          {df['bedrooms'].dtype}")
print(f"bathrooms dtype:         {df['bathrooms'].dtype}")
print(f"price_egp dtype:         {df['price_egp'].dtype}")
print(f"area_value dtype:        {df['area_value'].dtype}")
print(f"\nfurnished values:        {df['furnished'].unique()}")
print(f"completion_status vals:  {df['completion_status'].unique()}")

CONSISTENCY SUMMARY

bedrooms dtype:          Int64
bathrooms dtype:         Int64
price_egp dtype:         float64
area_value dtype:        float64

furnished values:        <StringArray>
['PARTLY', nan, 'Furnished', 'Unfurnished']
Length: 4, dtype: str
completion_status vals:  <StringArray>
[          'Off Plan',              'Ready',  'completed_primary',
   'off_plan_primary',                  nan, 'under-construction']
Length: 6, dtype: str


## Step 3: Completeness

### Reporting Missingness

In [47]:
def audit_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Produce a full missing value report sorted by missingness descending.

    Args:
        df: Input DataFrame.

    Returns:
        DataFrame with columns: column, missing_count, missing_pct, dtype.
    """
    missing_counts = df.isna().sum()
    missing_pcts   = (df.isna().mean() * 100).round(2)

    report = pd.DataFrame({
        "column":        missing_counts.index,
        "missing_count": missing_counts.values,
        "missing_pct":   missing_pcts.values,
        "dtype":         df.dtypes.values,
    })

    return report[report["missing_count"] > 0].sort_values("missing_pct", ascending=False)


missing_report = audit_missing_values(df)
print(f"Columns with missing values: {len(missing_report)}")
missing_report

Columns with missing values: 22


,column,missing_count,missing_pct,dtype
36,rera,35547,100.00,float64
34,video_url,35422,99.65,str
44,agent_languages,33400,93.96,str
47,broker_email,26051,73.29,str
10,price_period,26015,73.18,str
37,description,26015,73.18,str
42,agent_email,26015,73.18,str
39,payment_plan,26015,73.18,str
30,is_exclusive,26015,73.18,boolean
16,subdistrict,20120,56.60,str


### Dropping high-missingness and non-feature/irrelevant Columns

In [48]:
# Drop high-missingness or irrelevant columns
cols_to_drop = [
    "rera",                  # 100% missing
    "video_url",             # 99.6% missing
    "agent_languages",       # 93.7% missing
    "description",           # 71.6% missing (too large for cleaning)
    "broker_email",          # 71.7% missing
    "agent_email",           # 71.6% missing
    "payment_plan",          # 71.6% missing
    "price_period",          # 71.6% missing (mostly "sell")
    "title",
    "title.1",               # 28.4% missing (duplicate of title)
    "location_full",
    "scraped_at.1",          # 28.4% missing (duplicate of scraped_at)
    "detail_url",
    "url",                   # 28.4% missing (duplicate of detail_url)
    "id",                    # 28.4% missing (duplicate of listing_id)
    "reference",             # Low utility
    "has_video",             # Highly skewed
    "source",               
    "reference",
    "area_unit",
    "price_currency",
    "offering_type",
    "property_type",
    "listing_type",
    "category",
    "contact_phone",
    "contact_email",
    "contact_whatsapp",
    "agent_name",
    "agent_id",
    "agent_is_verified",
    "broker_name",
    "broker_phone",
    "broker_id",
    "scraped_at",
    "listed_date",
    "subdistrict",
    "is_exclusive",
    "internal_id",
    "listing_id",
    "lat",
    "lon",
    "is_verified",
    "is_new_construction" #100% false
]

df = df.drop(columns=cols_to_drop)
log_cleaning_action(
    step="Completeness",
    rule=f"Drop {len(cols_to_drop)} high-missingness or non-feature columns",
    records_affected=len(df),
    action=f"Dropped columns: {cols_to_drop}",
    rationale="Columns with >50% missing AND not required as model features",
)
print(f"Shape after column drops: {df.shape}")
print(f"Dropped {len(cols_to_drop)} columns")

[LOG] Completeness | Drop 44 high-missingness or non-feature columns | 35547 records | Dropped columns: ['rera', 'video_url', 'agent_languages', 'description', 'broker_email', 'agent_email', 'payment_plan', 'price_period', 'title', 'title.1', 'location_full', 'scraped_at.1', 'detail_url', 'url', 'id', 'reference', 'has_video', 'source', 'reference', 'area_unit', 'price_currency', 'offering_type', 'property_type', 'listing_type', 'category', 'contact_phone', 'contact_email', 'contact_whatsapp', 'agent_name', 'agent_id', 'agent_is_verified', 'broker_name', 'broker_phone', 'broker_id', 'scraped_at', 'listed_date', 'subdistrict', 'is_exclusive', 'internal_id', 'listing_id', 'lat', 'lon', 'is_verified', 'is_new_construction']
Shape after column drops: (35547, 27)
Dropped 44 columns


### Imputing Numeric Columns

In [49]:
NUMERIC_MEDIAN_IMPUTE_COLS = ["bedrooms", "bathrooms", "area_value"]


def impute_numeric_median(
    df: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    """Impute missing values in numeric columns using the column median.

    Use for: normally distributed or mildly skewed numeric features
    with <5% missingness and MCAR pattern.
    """
    df = df.copy()
    for col in columns:
        if col not in df.columns:
            continue
        n_missing = int(df[col].isna().sum())
        if n_missing == 0:
            continue
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        log_cleaning_action(
            step="Completeness",
            rule=f"{col}: median imputation",
            records_affected=n_missing,
            action=f"Fill NaN with median ({median_val:.2f})",
            rationale="MCAR pattern, <5% missing, critical feature — median robust to skew",
        )
    return df


df = impute_numeric_median(df, NUMERIC_MEDIAN_IMPUTE_COLS)

# Re-cast bedrooms and bathrooms to Int64 after median fill 
df["bedrooms"] = df["bedrooms"].round().astype("Int64")
df["bathrooms"] = df["bathrooms"].round().astype("Int64")

print(df[[c for c in NUMERIC_MEDIAN_IMPUTE_COLS if c in df.columns]].isna().sum())

[LOG] Completeness | bathrooms: median imputation | 1 records | Fill NaN with median (2.00)
bedrooms      0
bathrooms     0
area_value    0
dtype: int64


### Imputing District Column

In [50]:
def impute_district_by_location_group(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing district values using forward/backward fill within city-town groups.

    Logic:
    1. Within each (city, town) group, propagate known district values forward
       then backward to fill gaps.
    2. Any remaining NaN (entire group has no known district) -> fill with "Unknown".
    """
    df = df.copy()
    if "district" not in df.columns:
        print("No 'district' column found; skipping district imputation.")
        return df

    n_missing_before = int(df["district"].isna().sum())

    if ("city" in df.columns) and ("town" in df.columns):
        df["district"] = (
            df.groupby(["city", "town"])['district']
              .transform(lambda x: x.ffill().bfill())
        )
        n_filled_by_group = n_missing_before - int(df["district"].isna().sum())
    else:
        n_filled_by_group = 0

    # Remaining NaN -> "Unknown"
    df["district"] = df["district"].fillna("Unknown")

    log_cleaning_action(
        step="Completeness",
        rule="district: grouped ffill/bfill within city+town, then 'Unknown'",
        records_affected=n_missing_before,
        action=f"{n_filled_by_group} filled by group; rest -> 'Unknown'",
        rationale="MAR pattern: missingness depends on city/town, not the district value itself",
    )
    return df


df = impute_district_by_location_group(df)
print(f"district nulls remaining: {df['district'].isna().sum()}")

[LOG] Completeness | district: grouped ffill/bfill within city+town, then 'Unknown' | 5377 records | 4566 filled by group; rest -> 'Unknown'
district nulls remaining: 0


### Imputing Bool Columns

In [51]:
def add_missing_flag_and_impute_mode(
    df: pd.DataFrame,
    column: str,
    group_by: list[str] | None = None,
) -> pd.DataFrame:
    """Impute missing values with mode."""
    df = df.copy()
    n_missing = int(df[column].isna().sum())

    if group_by and all(col in df.columns for col in group_by):
        df[column] = df.groupby(group_by)[column].transform(
            lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
        )

    # Global fallback
    global_mode = df[column].mode()
    if not global_mode.empty:
        df[column] = df[column].fillna(global_mode[0])

    log_cleaning_action(
        step="Completeness",
        rule=f"{column}: mode imputation (group_by={group_by})",
        records_affected=n_missing,
        action="Filled with mode",
        rationale="Moderate missingness, the mode preserves the dominant pattern",
    )
    return df


# completion_status: impute by city mode (if column exists)
if "completion_status" in df.columns:
    df = add_missing_flag_and_impute_mode(df, "completion_status", group_by=["city"] if "city" in df.columns else None)

# furnished: create explicit 'Unknown' category
if "furnished" in df.columns:
    n_furnished_missing = int(df["furnished"].isna().sum())
    df["furnished"] = df["furnished"].fillna("Unknown")
    log_cleaning_action(
        step="Completeness",
        rule="furnished: fill NaN with 'Unknown' category",
        records_affected=n_furnished_missing,
        action="fillna('Unknown')",
        rationale=(
            "Missingness is informative — agents who don't disclose furnished status "
            "may represent a distinct listing pattern. 'Unknown' lets the model learn from it."
        ),
    )

print(df[["completion_status", "furnished"]].isna().sum())

[LOG] Completeness | completion_status: mode imputation (group_by=['city']) | 6 records | Filled with mode


[LOG] Completeness | furnished: fill NaN with 'Unknown' category | 6941 records | fillna('Unknown')
completion_status    0
furnished            0
dtype: int64


In [52]:
print(f"\nTotal nulls remaining across all columns:")
print(df.isna().sum()[df.isna().sum() > 0])


Total nulls remaining across all columns:
Series([], dtype: int64)


### Completeness Step Summary

In [53]:
print("COMPLETENESS CLEANING SUMMARY\n")
remaining_nulls = df.isna().sum()
columns_with_nulls = remaining_nulls[remaining_nulls > 0]
if columns_with_nulls.empty:
    print("No missing values remain in the dataset.")
else:
    print(f"Columns still with missing values: {len(columns_with_nulls)}")
    print(columns_with_nulls)
print(f"\nFinal shape: {df.shape}")

COMPLETENESS CLEANING SUMMARY

No missing values remain in the dataset.

Final shape: (35547, 27)


## Step 4: Uniqueness

### Removing Duplicates

In [54]:
def remove_exact_duplicates(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Remove exact duplicate rows.

    Args:
        df: Input DataFrame.
        subset: Columns to use as the duplicate key.

    Returns:
        Tuple of (df_deduped, df_removed_duplicates).
    """
    valid_subset = df.columns.to_list()

    duplicate_mask = df.duplicated(subset=valid_subset)
    df_removed = df[duplicate_mask].copy()
    df_deduped = df[~duplicate_mask].copy()

    log_cleaning_action(
        step="Uniqueness",
        rule=f"Exact duplicates on subset: {valid_subset}",
        records_affected=int(duplicate_mask.sum()),
        action=f"Keep first occurence. duplicates saved to interim/",
        rationale="Identical portal scrapes of the same listing inflate training data",
    )
    return df_deduped, df_removed

n_before = len(df)
df, df_removed_duplicates = remove_exact_duplicates(df)
n_after = len(df)

print(f"Records before dedup: {n_before:,}")
print(f"Records removed:      {n_before - n_after:,}")
print(f"Records remaining:    {n_after:,}")

# Save removed duplicates to interim for audit
Path("data/interim").mkdir(parents=True, exist_ok=True)
df_removed_duplicates.to_csv("data/interim/removed_duplicates.csv", index=False)
print("Saved removed duplicates to data/interim/removed_duplicates.csv")

[LOG] Uniqueness | Exact duplicates on subset: ['completion_status', 'price_egp', 'city', 'town', 'district', 'bedrooms', 'bathrooms', 'area_value', 'furnished', 'listing_level', 'is_premium', 'is_featured', 'is_direct_from_developer', 'images_count', 'amenities', 'dist_nearest_school_km', 'school_count_within_3km', 'dist_nearest_hospital_km', 'hospital_count_within_3km', 'dist_nearest_supermarket_km', 'supermarket_count_within_3km', 'dist_nearest_mall_km', 'mall_count_within_3km', 'dist_nearest_transit_station_km', 'transit_station_count_within_3km', 'dist_nearest_cafe_restaurant_km', 'cafe_restaurant_count_within_3km'] | 8624 records | Keep first occurence. duplicates saved to interim/
Records before dedup: 35,547
Records removed:      8,624
Records remaining:    26,923
Saved removed duplicates to data/interim/removed_duplicates.csv


### Uniquess Step Summary

In [55]:
print("UNIQUENESS CLEANING SUMMARY\n")
print(f"Exact duplicates removed:           {len(df_removed_duplicates):,}")
print(f"Records after deduplication:        {len(df):,}")

UNIQUENESS CLEANING SUMMARY

Exact duplicates removed:           8,624
Records after deduplication:        26,923


## Step 5: Outliers

### Detect Outliers Using IQR

In [56]:
def detect_outliers_iqr(
    series: pd.Series,
    multiplier: float = IQR_MULTIPLIER,
) -> pd.Series:
    """Return a boolean mask: True where the value is an IQR outlier.

    Args:
        series: Numeric pandas Series to check.
        multiplier: IQR multiplier for fence calculation (default 1.5).

    Returns:
        Boolean Series, True where value is outside [Q1 - m*IQR, Q3 + m*IQR].
    """
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - multiplier * iqr
    upper_fence = q3 + multiplier * iqr
    return (series < lower_fence) | (series > upper_fence)


outlier_target_cols = ["price_egp", "area_value"] + DISTANCE_COLUMNS + COUNT_COLUMNS
blocked_columns = set(globals().get("cols_to_drop", []))
valid_outlier_cols = [c for c in outlier_target_cols if c in df.columns and c not in blocked_columns]
outlier_audit = {}

for col in valid_outlier_cols:
    mask = detect_outliers_iqr(df[col].dropna())
    outlier_audit[col] = int(mask.sum())

outlier_audit_df = (
    pd.DataFrame.from_dict(outlier_audit, orient="index", columns=["iqr_outliers"])
    .sort_values("iqr_outliers", ascending=False)
)
print("IQR Outlier Counts (confirming Phase 1 findings):")
outlier_audit_df

IQR Outlier Counts (confirming Phase 1 findings):


,iqr_outliers
supermarket_count_within_3km,3558
cafe_restaurant_count_within_3km,3148
transit_station_count_within_3km,2703
dist_nearest_school_km,2419
school_count_within_3km,2359
dist_nearest_cafe_restaurant_km,2239
hospital_count_within_3km,2239
dist_nearest_supermarket_km,2133
dist_nearest_hospital_km,1922
mall_count_within_3km,1614


### Cap columns at percentile thresholds

In [57]:
def cap_column(
    df: pd.DataFrame,
    column: str,
    lower_percentile: float,
    upper_percentile: float,
) -> pd.DataFrame:
    """Cap a numeric column's values at specified percentile thresholds.

    Args:
        df: Input DataFrame.
        column: Column to cap.
        lower_percentile: Lower bound percentile (e.g., 0.02 for P2).
        upper_percentile: Upper bound percentile (e.g., 0.98 for P98).

    Returns:
        DataFrame with column values clipped between computed bounds.
    """
    df = df.copy()
    lower_cap = df[column].quantile(lower_percentile)
    upper_cap = df[column].quantile(upper_percentile)
    n_below = int((df[column] < lower_cap).sum())
    n_above = int((df[column] > upper_cap).sum())
    df[column] = df[column].clip(lower=lower_cap, upper=upper_cap)
    log_cleaning_action(
        step="Outliers",
        rule=f"{column}: cap at P{int(lower_percentile*100)}–P{int(upper_percentile*100)}",
        records_affected=n_below + n_above,
        action=f"Clip to [{lower_cap:,.0f}, {upper_cap:,.0f}]",
        rationale=f"{n_below} below lower cap, {n_above} above upper cap — capping preserves dataset size",
    )
    return df


df = cap_column(df, "price_egp", PRICE_CAP_LOWER_PERCENTILE, PRICE_CAP_UPPER_PERCENTILE)
print(f"price_egp range after capping: {df['price_egp'].min():,.0f} – {df['price_egp'].max():,.0f}")

[LOG] Outliers | price_egp: cap at P2–P98 | 1078 records | Clip to [1,573,968, 21,919,586]
price_egp range after capping: 1,573,968 – 21,919,586


In [58]:
df = cap_column(df, "area_value", AREA_CAP_LOWER_PERCENTILE, AREA_CAP_UPPER_PERCENTILE)

# Re-run cross-field validation: after capping, some records may still violate
# the bedroom/area ratio rule. Quarantine these as well.
if "bedrooms" in df.columns and "area_value" in df.columns:
    if "df_quarantine" not in globals():
        df_quarantine = pd.DataFrame(columns=list(df.columns) + ["rejection_reason"])

    area_bedroom_violation = (
        df["bedrooms"].astype(float, errors="ignore") >= 3
    ) & (df["area_value"] < (df["bedrooms"].astype(float, errors="ignore") * MIN_AREA_PER_BEDROOM))

    n_new_violations = int(area_bedroom_violation.sum())
    if n_new_violations > 0:
        new_quarantine = df[area_bedroom_violation].copy()
        new_quarantine["rejection_reason"] = "area_bedroom_ratio_implausible_post_capping"
        df_quarantine = pd.concat([df_quarantine, new_quarantine], ignore_index=True)
        df = df[~area_bedroom_violation].copy()
        log_cleaning_action(
            step="Outliers",
            rule="Cross-field re-validation post area capping",
            records_affected=n_new_violations,
            action="Quarantined records with implausible area/bedroom ratio",
            rationale="Area capping may expose previously hidden ratio violations",
        )

print(f"area_value range after capping: {df['area_value'].min():.1f} – {df['area_value'].max():.1f} sqm")
print(f"Updated quarantine size: {len(df_quarantine):,}")

[LOG] Outliers | area_value: cap at P1–P99 | 526 records | Clip to [40, 320]
area_value range after capping: 40.0 – 320.0 sqm
Updated quarantine size: 1,456


In [59]:
def cap_columns_at_percentile(
    df: pd.DataFrame,
    columns: list[str],
    upper_percentile: float,
) -> pd.DataFrame:
    """Cap multiple columns at the same upper percentile threshold.

    Args:
        df: Input DataFrame.
        columns: List of column names to cap.
        upper_percentile: Upper bound percentile.

    Returns:
        DataFrame with all specified columns capped.
    """
    df = df.copy()
    for col in columns:
        if col not in df.columns:
            continue
        df = cap_column(df, col, lower_percentile=0.0, upper_percentile=upper_percentile)
    return df


blocked_columns = set(globals().get("cols_to_drop", []))
valid_dist_cols = [c for c in DISTANCE_COLUMNS if c in df.columns and c not in blocked_columns]
df = cap_columns_at_percentile(df, valid_dist_cols, POI_DIST_CAP_PERCENTILE)

print("Distance column ranges after capping:")
for col in valid_dist_cols:
    print(f"  {col}: {df[col].min():.3f} – {df[col].max():.3f} km")

[LOG] Outliers | dist_nearest_school_km: cap at P0–P99 | 245 records | Clip to [0, 52]
[LOG] Outliers | dist_nearest_hospital_km: cap at P0–P99 | 267 records | Clip to [0, 60]
[LOG] Outliers | dist_nearest_supermarket_km: cap at P0–P99 | 270 records | Clip to [0, 17]
[LOG] Outliers | dist_nearest_mall_km: cap at P0–P99 | 250 records | Clip to [0, 99]
[LOG] Outliers | dist_nearest_transit_station_km: cap at P0–P99 | 220 records | Clip to [0, 19]
[LOG] Outliers | dist_nearest_cafe_restaurant_km: cap at P0–P99 | 187 records | Clip to [0, 20]
Distance column ranges after capping:
  dist_nearest_school_km: 0.041 – 51.864 km
  dist_nearest_hospital_km: 0.010 – 59.521 km
  dist_nearest_supermarket_km: 0.009 – 16.920 km
  dist_nearest_mall_km: 0.066 – 99.012 km
  dist_nearest_transit_station_km: 0.013 – 19.397 km
  dist_nearest_cafe_restaurant_km: 0.002 – 20.291 km


### Outlier Step Summary

In [60]:
print("OUTLIER TREATMENT SUMMARY\n")
print(f"price_egp:   capped at P{int(PRICE_CAP_LOWER_PERCENTILE*100)}–P{int(PRICE_CAP_UPPER_PERCENTILE*100)}")
print(f"area_value:  capped at P{int(AREA_CAP_LOWER_PERCENTILE*100)}–P{int(AREA_CAP_UPPER_PERCENTILE*100)}")
print(f"dist_* cols: capped at P{int(POI_DIST_CAP_PERCENTILE*100)}")
print(f"\nFinal shape: {df.shape}")

OUTLIER TREATMENT SUMMARY

price_egp:   capped at P2–P98
area_value:  capped at P1–P99
dist_* cols: capped at P99

Final shape: (26923, 27)


## Target Re-derivation

`price_category` must be re-derived from the cleaned `price_egp` after:
- Accuracy corrections (prices set to NaN or corrected)
- Price capping (P2–P98)

Using the same design from Phase 1: quantile binning into 3 equal tiers ensures ~33.3% per class.

In [61]:
def derive_price_category(df: pd.DataFrame, price_col: str = "price_egp") -> pd.DataFrame:
    """Derive the price_category target variable from cleaned price using quantile binning.

    Bins:
    - Low:    bottom 33.3% by price
    - Medium: middle 33.3% by price
    - High:   top 33.3% by price

    Args:
        df: Input DataFrame with a clean numeric price column.
        price_col: Name of the price column.

    Returns:
        DataFrame with 'price_category' column added (dtype: category).

    Raises:
        ValueError: If price_col contains NaN values (must be imputed first).
    """
    if df[price_col].isna().any():
        raise ValueError(
            f"'{price_col}' still contains NaN. Impute before deriving target."
        )

    df = df.copy()
    df["price_category"] = pd.qcut(
        df[price_col],
        q=3,
        labels=["Low", "Medium", "High"],
        duplicates="drop",
    )

    log_cleaning_action(
        step="Target",
        rule="price_category derived from cleaned price_egp via quantile binning",
        records_affected=len(df),
        action="pd.qcut(q=3, labels=['Low','Medium','High'])",
        rationale="Re-derive after accuracy corrections and capping to ensure consistency",
    )
    return df


# Handle any remaining NaN in price_egp before target derivation
n_price_null = int(df["price_egp"].isna().sum())
if n_price_null > 0:
    price_median = df["price_egp"].median()
    df["price_egp"] = df["price_egp"].fillna(price_median)
    log_cleaning_action(
        step="Completeness",
        rule="price_egp: median imputation for remaining nulls",
        records_affected=n_price_null,
        action=f"fillna({price_median:,.0f})",
        rationale="Required before target derivation; MCAR assumption",
    )

df = derive_price_category(df)

print("price_category distribution:")
print(df["price_category"].value_counts())
print(df["price_category"].value_counts(normalize=True).round(3))

[LOG] Target | price_category derived from cleaned price_egp via quantile binning | 26923 records | pd.qcut(q=3, labels=['Low','Medium','High'])
price_category distribution:
price_category
Low       9063
Medium    9044
High      8816
Name: count, dtype: int64
price_category
Low       0.337
Medium    0.336
High      0.327
Name: proportion, dtype: float64


In [62]:
# Range checks
if "area_value" in df.columns:
    assert (df["area_value"] >= AREA_MIN_SQM).all(), "area_value below minimum"
if "price_egp" in df.columns:
    assert (df["price_egp"] > 0).all(), "price_egp has non-positive values"
if "latitude" in df.columns:
    assert df["latitude"].between(LAT_MIN, LAT_MAX).all(), "latitude out of Egypt bounds"
if "longitude" in df.columns:
    assert df["longitude"].between(LON_MIN, LON_MAX).all(), "longitude out of Egypt bounds"
print("All range assertions passed")

All range assertions passed


## Save Outputs

In [63]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Saved clean data → {PROCESSED_DATA_PATH}")
print(f"Shape: {df.shape}")

Saved clean data → data\processed\clean_listings.csv
Shape: (26923, 28)


In [64]:
df_quarantine.to_csv(QUARANTINE_DATA_PATH, index=False)
print(f"Saved quarantine → {QUARANTINE_DATA_PATH}")
print(f"Quarantine shape: {df_quarantine.shape}")
# print(f"\nRejection reason breakdown:")
# print(df_quarantine["rejection_reason"].value_counts())

Saved quarantine → data\processed\quarantine_listings.csv
Quarantine shape: (1456, 70)


In [65]:
Path("reports").mkdir(parents=True, exist_ok=True)
if cleaning_log:
    cleaning_log_df = pd.DataFrame(cleaning_log)
    cleaning_log_df.to_csv(CLEANING_LOG_PATH, index=False)
    print(f"Saved cleaning log → {CLEANING_LOG_PATH}")
    print(f"Total log entries: {len(cleaning_log_df)}")
    cleaning_log_df
else:
    print("No cleaning log entries to save")

Saved cleaning log → reports\cleaning_log.csv
Total log entries: 39


In [66]:
print("DATA CLEANING SUMMARY\n")
print(f"Raw records:               {len(raw_df):>8,}")
print(f"Quarantined records:       {len(df_quarantine):>8,}  ({len(df_quarantine)/len(raw_df)*100:.1f}%)")
print(f"Duplicates removed:        {len(df_removed_duplicates):>8,}  ({len(df_removed_duplicates)/len(raw_df)*100:.1f}%)")
print(f"Clean records:             {len(df):>8,}  ({len(df)/len(raw_df)*100:.1f}%)")
print()
print(f"Original columns:          {len(raw_df.columns):>8}")
print(f"Columns dropped:           {len(cols_to_drop):>8}")
print(f"New columns added:         {len(df.columns) - (len(raw_df.columns) - len(cols_to_drop)):>8}")
print(f"Final columns:             {len(df.columns):>8}")
print()
print(f"Target distribution:")
dist = df["price_category"].value_counts()
for cls in ["Low", "Medium", "High"]:
    count = dist.get(cls, 0)
    print(f"  {cls:<10}: {count:>6,}  ({count/len(df)*100:.1f}%)")
print()
print(f"Outputs saved:")
print(PROCESSED_DATA_PATH)
print(QUARANTINE_DATA_PATH)
print(CLEANING_LOG_PATH)

DATA CLEANING SUMMARY

Raw records:                 37,003
Quarantined records:          1,456  (3.9%)
Duplicates removed:           8,624  (23.3%)
Clean records:               26,923  (72.8%)

Original columns:                70
Columns dropped:                 44
New columns added:                2
Final columns:                   28

Target distribution:
  Low       :  9,063  (33.7%)
  Medium    :  9,044  (33.6%)
  High      :  8,816  (32.7%)

Outputs saved:
data\processed\clean_listings.csv
data\processed\quarantine_listings.csv
reports\cleaning_log.csv
